# Flood Probability Estimation using AutoML — Balikpapan & Samarinda

Implementasi paper **"Flood Probability Estimation using Automated Machine Learning (AutoML) in Balikpapan and Samarinda"** (IIAIR, Vol.1, Issue 1, 2026).

Notebook ini **memakai data asli** (tanpa data sintetis) dan mengikuti 5 tahap metodologi paper:

1. **Data Collection** — titik banjir DIBI (BNPB)
2. **Flood & Non-Flood Point Sampling** — binary classification, rasio 1:1
3. **Data Preprocessing & Feature Engineering** — standardisasi numerik + encoding land cover
4. **Machine Learning Modeling** — Random Forest, XGBoost, CatBoost dalam kerangka **AutoML (FLAML)**
5. **Model Evaluation** — Accuracy, Precision, Recall, F1, **AUC-ROC** (utama)

Lalu menghasilkan **peta probabilitas banjir** (GeoTIFF + PNG).

> Urutan menjalankan: jalankan sel dari atas ke bawah. Pada Sel **3 (Konfigurasi)** isi path file dan **nama kolom** sesuai datamu — semua titik yang perlu kamu sesuaikan ditandai komentar `# << SESUAIKAN`.

## 0. Setup environment

In [ ]:
!pip -q install flaml[automl] catboost xgboost shap scikit-learn 2>/dev/null
!pip -q install geopandas rasterio shapely pyproj rtree 2>/dev/null
print("Selesai instalasi.")

In [ ]:
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from flaml import AutoML

import geopandas as gpd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
from shapely.geometry import Point

try:
    import shap; HAS_SHAP = True
except Exception:
    HAS_SHAP = False

print("Import OK. SHAP:", HAS_SHAP)

## 1. Hubungkan data

**Pilihan A — Google Drive (disarankan, file tidak hilang saat sesi reset):**
Jalankan sel di bawah, lalu salin semua file dataset ke folder Drive-mu (mis. `MyDrive/flood_data/`).

**Pilihan B — Upload langsung:** lewati mount, jalankan sel upload, file masuk ke `/content/data`.

In [ ]:
# --- Pilihan A: Google Drive ---
from google.colab import drive
drive.mount("/content/drive")
# Setelah ini set DATA_DIR di Sel 3 ke folder Drive-mu, contoh:
#   DATA_DIR = "/content/drive/MyDrive/flood_data"

In [ ]:
# --- Pilihan B: Upload manual (jalankan jika TIDAK pakai Drive) ---
# import os
# os.makedirs("/content/data", exist_ok=True)
# from google.colab import files
# up = files.upload()                 # pilih semua file dataset
# for fn in up:
#     os.replace(fn, os.path.join("/content/data", fn))
# print(os.listdir("/content/data"))

## 2. Sumber data & layout file yang dibutuhkan

| Faktor | Sumber | File yang disiapkan |
|--------|--------|---------------------|
| Titik banjir (label 1) | DIBI – BNPB (dibi.bnpb.go.id) | `dibi_flood.csv` (punya kolom nama wilayah) **atau** sudah berisi `lon,lat` |
| Elevation, Slope | SRTM 30 m DEM | `dem_srtm.tif` (boleh masih lon/lat — notebook reproyeksi otomatis) |
| Sungai | RBI – BIG (tanahair.indonesia.go.id) | `rbi_sungai.shp` (+ .dbf/.shx/.prj) |
| Jalan | RBI – BIG | `rbi_jalan.shp` |
| Land cover | RBI – BIG | `rbi_landcover.shp` (punya kolom kelas) |
| Batas kecamatan | RBI – BIG | `rbi_admin.shp` (punya kolom nama) |
| Penduduk | BPS Balikpapan & Kaltim | `bps_penduduk.csv` (kolom nama kecamatan + jumlah penduduk) |

> Shapefile selalu terdiri dari beberapa berkas (`.shp .shx .dbf .prj`). Upload **semuanya**.

## 3. Konfigurasi — isi bagian ini

In [ ]:
# ============================ KONFIGURASI ============================
DATA_DIR = "/content/data"          # << SESUAIKAN (mis. "/content/drive/MyDrive/flood_data")
OUT_DIR  = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
TIME_BUDGET = 120                   # anggaran waktu AutoML FLAML (detik)
UTM_CRS = "EPSG:32750"              # WGS84 / UTM zone 50S (Kaltim) -> satuan meter
WGS84   = "EPSG:4326"

# ---- Path file ----
DEM_PATH       = os.path.join(DATA_DIR, "dem_srtm.tif")
RIVER_PATH     = os.path.join(DATA_DIR, "rbi_sungai.shp")
ROAD_PATH      = os.path.join(DATA_DIR, "rbi_jalan.shp")
LANDCOVER_PATH = os.path.join(DATA_DIR, "rbi_landcover.shp")
ADMIN_PATH     = os.path.join(DATA_DIR, "rbi_admin.shp")
BPS_PATH       = os.path.join(DATA_DIR, "bps_penduduk.csv")
DIBI_PATH      = os.path.join(DATA_DIR, "dibi_flood.csv")

# ---- Cara memperoleh titik banjir ----
FLOOD_POINTS_MODE = "geocode"       # "geocode" = dari nama wilayah DIBI ; "lonlat" = CSV sudah ada kolom lon,lat
DIBI_NAME_COL  = "Kelurahan"        # << SESUAIKAN kolom nama wilayah di dibi_flood.csv
ADMIN_NAME_COL = "NAMOBJ"           # << SESUAIKAN kolom nama wilayah di rbi_admin.shp
DIBI_LON_COL   = "lon"              # dipakai jika FLOOD_POINTS_MODE = "lonlat"
DIBI_LAT_COL   = "lat"

# ---- Nama kolom layer lain ----
LC_CLASS_COL = "REMARK"             # << SESUAIKAN kolom kelas penutup lahan di rbi_landcover.shp
BPS_NAME_COL = "nama_kecamatan"     # << SESUAIKAN kolom nama kecamatan di bps_penduduk.csv
BPS_POP_COL  = "jumlah_penduduk"    # << SESUAIKAN kolom jumlah penduduk di bps_penduduk.csv

# ---- Sampling titik non-banjir ----
NONFLOOD_ELEV_QUANTILE = 0.5        # ambil dari zona elevasi >= median
NONFLOOD_MIN_DIST      = 750.0      # jarak minimum (m) dari titik banjir

NUMERIC_COLS = ["elevation", "slope", "dist_river", "dist_road", "pop_density"]
CAT_COLS     = ["landcover"]
FEATURE_COLS = NUMERIC_COLS + CAT_COLS
np.random.seed(RANDOM_SEED)
print("Konfigurasi dimuat. DATA_DIR =", DATA_DIR)

## 4. Fungsi pemrosesan geospasial

In [ ]:
def assert_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File tidak ditemukan: {path}\n-> Cek DATA_DIR & nama file di Sel 3, dan pastikan sudah terupload.")

def reproject_raster(src_path, dst_path, dst_crs=UTM_CRS):
    """Reproyeksi raster ke dst_crs (agar satuan meter). Mengembalikan dst_path."""
    with rasterio.open(src_path) as src:
        if str(src.crs) == dst_crs:
            return src_path
        transform, w, h = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
        meta = src.meta.copy()
        meta.update(crs=dst_crs, transform=transform, width=w, height=h)
        with rasterio.open(dst_path, "w", **meta) as dst:
            for i in range(1, src.count + 1):
                reproject(source=rasterio.band(src, i), destination=rasterio.band(dst, i),
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear)
    return dst_path

def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        return arr, src.transform, src.crs

def compute_slope_deg(dem, transform):
    px, py = abs(transform.a), abs(transform.e)     # ukuran piksel (meter setelah reproyeksi)
    dzdy, dzdx = np.gradient(dem, py, px)
    return np.degrees(np.arctan(np.sqrt(dzdx ** 2 + dzdy ** 2)))

def sample_array_at_points(points_gdf, array, transform):
    vals = []
    for geom in points_gdf.geometry:
        r, c = rowcol(transform, geom.x, geom.y)
        if 0 <= r < array.shape[0] and 0 <= c < array.shape[1]:
            vals.append(array[r, c])
        else:
            vals.append(np.nan)
    return np.array(vals, dtype=float)

def attach_polygon_value(points_gdf, poly_gdf, value_col):
    poly = poly_gdf.to_crs(UTM_CRS)[["geometry", value_col]]
    joined = gpd.sjoin(points_gdf, poly, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]
    return joined.loc[points_gdf.index, value_col].values

def generate_nonflood_points(flood_gdf, dem, transform, n,
                             elev_quantile=NONFLOOD_ELEV_QUANTILE,
                             min_dist=NONFLOOD_MIN_DIST, seed=RANDOM_SEED):
    """Titik non-banjir acak di zona elevasi lebih tinggi & jauh dari titik banjir (rasio 1:1)."""
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = flood_gdf.total_bounds
    pad = 0.15 * max(maxx - minx, maxy - miny)
    minx, miny, maxx, maxy = minx - pad, miny - pad, maxx + pad, maxy + pad
    thr = np.nanquantile(dem, elev_quantile)
    flood_union = flood_gdf.unary_union
    pts, tries = [], 0
    while len(pts) < n and tries < n * 300:
        tries += 1
        px, py = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        r, c = rowcol(transform, px, py)
        if not (0 <= r < dem.shape[0] and 0 <= c < dem.shape[1]):
            continue
        if np.isnan(dem[r, c]) or dem[r, c] < thr:
            continue
        p = Point(px, py)
        if p.distance(flood_union) < min_dist:
            continue
        pts.append(p)
    print(f"Titik non-banjir terbentuk: {len(pts)} (target {n})")
    return gpd.GeoDataFrame(geometry=pts, crs=UTM_CRS)

def _norm(s):
    return str(s).strip().lower()

def geocode_flood_from_admin(dibi_csv, admin_gdf, dibi_name_col, admin_name_col):
    """Ubah kejadian banjir DIBI (nama wilayah) -> titik = centroid poligon wilayah."""
    dibi = pd.read_csv(dibi_csv)
    admin = admin_gdf.to_crs(UTM_CRS).copy()
    admin["_key"] = admin[admin_name_col].map(_norm)
    admin["_cx"] = admin.geometry.centroid.x
    admin["_cy"] = admin.geometry.centroid.y
    dibi["_key"] = dibi[dibi_name_col].map(_norm)
    merged = dibi.merge(admin[["_key", "_cx", "_cy"]].drop_duplicates("_key"), on="_key", how="left")
    ok = merged.dropna(subset=["_cx", "_cy"]).reset_index(drop=True)
    print(f"Kejadian tercocokkan: {len(ok)}/{len(dibi)}")
    if len(ok) == 0:
        raise ValueError("Tidak ada nama wilayah yang cocok. Cek DIBI_NAME_COL & ADMIN_NAME_COL / penulisan nama.")
    geom = [Point(x, y) for x, y in zip(ok["_cx"], ok["_cy"])]
    return gpd.GeoDataFrame(ok.drop(columns=["_cx", "_cy"]), geometry=geom, crs=UTM_CRS)

print("Fungsi geospasial siap.")

## 5. Muat layer & ekstraksi faktor

### 5a. DEM → elevation & slope (reproyeksi otomatis ke UTM)

In [ ]:
assert_file(DEM_PATH)
dem_utm_path = os.path.join(OUT_DIR, "dem_utm.tif")
dem_utm_path = reproject_raster(DEM_PATH, dem_utm_path, UTM_CRS)
dem, dem_transform, dem_crs = load_raster(dem_utm_path)
slope = compute_slope_deg(dem, dem_transform)
print("DEM:", dem.shape, "| CRS:", dem_crs, "| elev range:",
      np.nanmin(dem).round(1), "-", np.nanmax(dem).round(1), "m")

### 5b. Sungai, jalan, land cover, batas kecamatan + kepadatan penduduk (BPS)

In [ ]:
for p in [RIVER_PATH, ROAD_PATH, LANDCOVER_PATH, ADMIN_PATH, BPS_PATH]:
    assert_file(p)

rivers    = gpd.read_file(RIVER_PATH).to_crs(UTM_CRS)
roads     = gpd.read_file(ROAD_PATH).to_crs(UTM_CRS)
landcover = gpd.read_file(LANDCOVER_PATH).to_crs(UTM_CRS)
admin     = gpd.read_file(ADMIN_PATH).to_crs(UTM_CRS)
bps       = pd.read_csv(BPS_PATH)

# Encoding land cover yang konsisten antara titik & peta
lc_classes = sorted(landcover[LC_CLASS_COL].dropna().astype(str).unique().tolist())
LC_MAP = {c: i for i, c in enumerate(lc_classes)}
print("Kelas land cover:", LC_MAP)

# Kepadatan penduduk = jumlah penduduk / luas kecamatan (km^2)
admin["_key"] = admin[ADMIN_NAME_COL].map(_norm)
bps["_key"]   = bps[BPS_NAME_COL].map(_norm)
admin = admin.merge(bps[["_key", BPS_POP_COL]].drop_duplicates("_key"), on="_key", how="left")
admin["area_km2"]    = admin.geometry.area / 1e6
admin["pop_density"] = admin[BPS_POP_COL] / admin["area_km2"]
print("Kecamatan dengan penduduk tercocokkan:", admin["pop_density"].notna().sum(), "/", len(admin))

### 5c. Titik banjir (label 1)

In [ ]:
assert_file(DIBI_PATH)
if FLOOD_POINTS_MODE == "geocode":
    flood_gdf = geocode_flood_from_admin(DIBI_PATH, admin, DIBI_NAME_COL, ADMIN_NAME_COL)
elif FLOOD_POINTS_MODE == "lonlat":
    d = pd.read_csv(DIBI_PATH)
    flood_gdf = gpd.GeoDataFrame(d, geometry=gpd.points_from_xy(d[DIBI_LON_COL], d[DIBI_LAT_COL]),
                                 crs=WGS84).to_crs(UTM_CRS)
else:
    raise ValueError("FLOOD_POINTS_MODE harus 'geocode' atau 'lonlat'.")
flood_gdf = flood_gdf[flood_gdf.geometry.notna()].reset_index(drop=True)
print("Jumlah titik banjir:", len(flood_gdf))

### 5d. Titik non-banjir (label 0, rasio 1:1) & gabung

In [ ]:
nonflood_gdf = generate_nonflood_points(flood_gdf, dem, dem_transform, n=len(flood_gdf))

flood_gdf = flood_gdf.copy();    flood_gdf["label"] = 1
nonflood_gdf = nonflood_gdf.copy(); nonflood_gdf["label"] = 0
pts = pd.concat([flood_gdf[["geometry", "label"]], nonflood_gdf[["geometry", "label"]]],
                ignore_index=True)
pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=UTM_CRS)
print("Total titik:", len(pts), "| distribusi:", pts["label"].value_counts().to_dict())

### 5e. Ekstraksi conditioning factor di tiap titik

In [ ]:
rivers_union = rivers.unary_union
roads_union  = roads.unary_union

pts["elevation"]  = sample_array_at_points(pts, dem, dem_transform)
pts["slope"]      = sample_array_at_points(pts, slope, dem_transform)
pts["dist_river"] = pts.geometry.apply(lambda g: g.distance(rivers_union))
pts["dist_road"]  = pts.geometry.apply(lambda g: g.distance(roads_union))

lc_raw = attach_polygon_value(pts, landcover, LC_CLASS_COL)
pts["landcover"]   = [LC_MAP.get(str(v), -1) for v in lc_raw]
pts["pop_density"] = attach_polygon_value(pts, admin, "pop_density")

df = pd.DataFrame(pts.drop(columns="geometry"))
df["x"] = pts.geometry.x.values
df["y"] = pts.geometry.y.values
print("Dataset berlabel:", df.shape)
df.head()

## 6. Eksplorasi singkat

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().rename({0: "Non-Banjir", 1: "Banjir"}).plot(
    kind="bar", ax=ax[0], color=["#4C9F70", "#D1495B"]); ax[0].set_title("Keseimbangan kelas"); ax[0].tick_params(axis="x", rotation=0)
corr = df[NUMERIC_COLS + ["label"]].corr()
im = ax[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax[1].set_xticks(range(len(corr))); ax[1].set_xticklabels(corr.columns, rotation=45, ha="right")
ax[1].set_yticks(range(len(corr))); ax[1].set_yticklabels(corr.columns); ax[1].set_title("Korelasi")
for (a, b), v in np.ndenumerate(corr.values): ax[1].text(b, a, f"{v:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax[1], fraction=0.046); plt.tight_layout(); plt.show()
df[NUMERIC_COLS].describe().round(2)

## 7. Preprocessing & Feature Engineering
Buang nilai hilang/invalid, standardisasi faktor numerik, encoding land cover.

In [ ]:
before = len(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLS + ["label"]).reset_index(drop=True)
print(f"Baris dibuang (nilai hilang/invalid): {before - len(df)} | tersisa: {len(df)}")

scaler  = StandardScaler().fit(df[NUMERIC_COLS])
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(df[CAT_COLS])

def transform_features(frame):
    Xn = pd.DataFrame(scaler.transform(frame[NUMERIC_COLS]), columns=NUMERIC_COLS, index=frame.index)
    Xc = pd.DataFrame(encoder.transform(frame[CAT_COLS]),    columns=CAT_COLS,     index=frame.index)
    return pd.concat([Xn, Xc], axis=1)

X = transform_features(df); y = df["label"].astype(int)
print("Matriks fitur:", X.shape)

## 8. Train / test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 9. Model individual: Random Forest, XGBoost, CatBoost

In [ ]:
models = {}
models["Random Forest"] = RandomForestClassifier(
    n_estimators=400, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)
models["XGBoost"] = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)
models["CatBoost"] = CatBoostClassifier(
    iterations=400, depth=6, learning_rate=0.05, random_seed=RANDOM_SEED, verbose=0).fit(X_train, y_train)
print("Terlatih:", list(models.keys()))

## 10. AutoML (FLAML)
Pemilihan model + tuning hyperparameter otomatis di antara RF/XGBoost/CatBoost, dioptimasi ke **ROC-AUC**.

In [ ]:
automl = AutoML()
settings = dict(task="classification", metric="roc_auc",
                estimator_list=["rf", "xgboost", "catboost"],
                time_budget=TIME_BUDGET, eval_method="cv", n_splits=5,
                seed=RANDOM_SEED, verbose=1)
try:
    automl.fit(X_train=X_train, y_train=y_train, **settings)
except Exception as e:
    print("Ulang tanpa catboost:", e)
    settings["estimator_list"] = ["rf", "xgboost"]
    automl.fit(X_train=X_train, y_train=y_train, **settings)
print("\nEstimator terbaik :", automl.best_estimator)
print("Konfigurasi terbaik:", automl.best_config)
models["AutoML (FLAML)"] = automl

## 11. Evaluasi

In [ ]:
def get_proba(model, Xte):
    return np.asarray(model.predict_proba(Xte))[:, 1]

def evaluate_model(name, model, Xte, yte):
    proba = get_proba(model, Xte); pred = (proba >= 0.5).astype(int)
    return dict(Model=name, Accuracy=accuracy_score(yte, pred),
                Precision=precision_score(yte, pred, zero_division=0),
                Recall=recall_score(yte, pred, zero_division=0),
                F1=f1_score(yte, pred, zero_division=0),
                AUC_ROC=roc_auc_score(yte, proba)), proba

results, probas = [], {}
for name, m in models.items():
    metr, pr = evaluate_model(name, m, X_test, y_test); results.append(metr); probas[name] = pr
res_df = pd.DataFrame(results).set_index("Model").sort_values("AUC_ROC", ascending=False)
best_name = res_df.index[0]; best_model = models[best_name]
print("Model terbaik (AUC-ROC tertinggi):", best_name)
res_df.round(4)

### 11a. Kurva ROC

In [ ]:
plt.figure(figsize=(7, 6))
for name, pr in probas.items():
    fpr, tpr, _ = roc_curve(y_test, pr)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_test, pr):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Kurva ROC"); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 11b. Confusion matrix & laporan model terbaik

In [ ]:
best_pred = (get_proba(best_model, X_test) >= 0.5).astype(int)
cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(4.5, 4)); ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-Banjir", "Banjir"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Non-Banjir", "Banjir"])
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title(f"Confusion Matrix — {best_name}")
for (a, b), v in np.ndenumerate(cm):
    ax.text(b, a, str(v), ha="center", va="center", color="white" if v > cm.max()/2 else "black")
plt.tight_layout(); plt.show()
print(classification_report(y_test, best_pred, target_names=["Non-Banjir", "Banjir"]))

## 12. Feature importance

In [ ]:
frames = []
for name, m in models.items():
    fi = None
    if hasattr(m, "feature_importances_"):
        fi = np.asarray(m.feature_importances_)
    elif name == "AutoML (FLAML)":
        est = getattr(getattr(m, "model", None), "estimator", None)
        if est is not None and hasattr(est, "feature_importances_"):
            fi = np.asarray(est.feature_importances_)
    if fi is not None and len(fi) == len(FEATURE_COLS):
        frames.append(pd.Series(fi / fi.sum(), index=FEATURE_COLS, name=name))
if frames:
    imp = pd.concat(frames, axis=1); imp["mean"] = imp.mean(axis=1); imp = imp.sort_values("mean")
    imp["mean"].plot(kind="barh", figsize=(7, 4), color="#2A6F97"); plt.title("Feature importance rata-rata")
    plt.xlabel("Importance"); plt.tight_layout(); plt.show()
    display(imp.round(3))

### 12a. SHAP (opsional)

In [ ]:
if HAS_SHAP:
    try:
        cand = [n for n in ["CatBoost", "XGBoost", "Random Forest"] if n in models]
        sname = max(cand, key=lambda n: res_df.loc[n, "AUC_ROC"])
        expl = shap.TreeExplainer(models[sname])
        samp = X_test.sample(min(300, len(X_test)), random_state=RANDOM_SEED)
        sv = expl.shap_values(samp)
        if isinstance(sv, list): sv = sv[1]
        shap.summary_plot(sv, samp, feature_names=FEATURE_COLS, show=True)
        print("SHAP model:", sname)
    except Exception as e:
        print("SHAP dilewati:", e)

## 13. Peta probabilitas banjir (GeoTIFF + PNG)
Membangun stack faktor selaras grid DEM, memprediksi probabilitas tiap sel, dan menyimpan peta.

In [ ]:
H, W = dem.shape
px = abs(dem_transform.a)

# Distance fields via rasterisasi + distance transform (cepat untuk seluruh grid)
river_mask = rasterize(((g, 1) for g in rivers.geometry), out_shape=(H, W),
                       transform=dem_transform, fill=0, all_touched=True)
road_mask  = rasterize(((g, 1) for g in roads.geometry),  out_shape=(H, W),
                       transform=dem_transform, fill=0, all_touched=True)
dist_river_grid = distance_transform_edt(river_mask == 0) * px
dist_road_grid  = distance_transform_edt(road_mask == 0) * px

# Land cover grid (kode konsisten dgn LC_MAP) & pop density grid
lc_grid = rasterize(((g, LC_MAP.get(str(v), -1)) for g, v in zip(landcover.geometry, landcover[LC_CLASS_COL])),
                    out_shape=(H, W), transform=dem_transform, fill=-1, dtype="float32")
pop_grid = rasterize(((g, float(v)) for g, v in zip(admin.geometry, admin["pop_density"]) if pd.notna(v)),
                     out_shape=(H, W), transform=dem_transform, fill=np.nan, dtype="float32")

grid = pd.DataFrame({
    "elevation":   dem.ravel(),
    "slope":       slope.ravel(),
    "dist_river":  dist_river_grid.ravel(),
    "dist_road":   dist_road_grid.ravel(),
    "pop_density": pop_grid.ravel(),
    "landcover":   lc_grid.ravel().astype(int),
})
valid = grid[NUMERIC_COLS].notna().all(axis=1).values
prob_flat = np.full(len(grid), np.nan)
if valid.sum() > 0:
    prob_flat[valid] = get_proba(best_model, transform_features(grid[valid]))
prob_map = prob_flat.reshape(H, W)

# Simpan GeoTIFF
tif_out = os.path.join(OUT_DIR, "flood_probability_map.tif")
with rasterio.open(tif_out, "w", driver="GTiff", height=H, width=W, count=1,
                   dtype="float32", crs=UTM_CRS, transform=dem_transform, nodata=np.nan) as dst:
    dst.write(prob_map.astype("float32"), 1)

# Plot PNG
plt.figure(figsize=(8, 8))
im = plt.imshow(prob_map, cmap="turbo", vmin=0, vmax=1)
plt.colorbar(im, label="Probabilitas banjir")
plt.title(f"Peta Probabilitas Banjir — {best_name}"); plt.axis("off")
png_out = os.path.join(OUT_DIR, "flood_probability_map.png")
plt.savefig(png_out, dpi=150, bbox_inches="tight"); plt.show()
print("Tersimpan:", tif_out, "&", png_out)

## 14. Simpan hasil

In [ ]:
res_df.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"))
df.to_csv(os.path.join(OUT_DIR, "labelled_points.csv"), index=False)
print("Output di", OUT_DIR, ":", os.listdir(OUT_DIR))
# Unduh (opsional):
# from google.colab import files
# for f in ["model_comparison.csv","labelled_points.csv","flood_probability_map.tif","flood_probability_map.png"]:
#     files.download(os.path.join(OUT_DIR, f))

## Catatan
- Model terbaik dipilih berdasarkan **AUC-ROC** (sesuai paper), didampingi Precision/Recall/F1.
- Jika kepadatan penduduk banyak yang `NaN`, periksa pencocokan nama di `ADMIN_NAME_COL` vs `BPS_NAME_COL` (ejaan/spasi).
- Jika geocoding DIBI sedikit yang cocok, samakan level wilayah (DIBI sering memakai kelurahan; pastikan `rbi_admin.shp` juga level kelurahan).
- DEM otomatis direproyeksi ke UTM 50S agar slope & jarak bersatuan meter.